In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
import pandas as pd

base_path = '/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset/Exports'
dataframes = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.lower().endswith('.csv'):
            file_path = os.path.join(root, file)

            try:
                # pular a primeira linha (Domestic exports)
                df = pd.read_csv(file_path, skiprows=1)

                # limpar espaços nos nomes das colunas
                df.columns = df.columns.str.strip()

                # remover linhas onde Period não é data válida
                df['Period'] = pd.to_datetime(df['Period'], errors='coerce')
                df = df[df['Period'].notna()]

                # converter colunas numéricas
                df['Value ($)'] = pd.to_numeric(df['Value ($)'], errors='coerce')
                df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

                # remover linhas sem valores principais
                df = df[df['Value ($)'].notna()]

                # adicionar coluna de origem
                df["source_file"] = file

                dataframes.append(df)

            except Exception as e:
                print(f"Failed to read {file_path}: {e}")

# concatenar todos os arquivos
if dataframes:
    dataset = pd.concat(dataframes, ignore_index=True)
    print("Final concatenated dataset shape:", dataset.shape)
else:
    print("No CSV files found.")

# salvar parquet
output_path = "/content/drive/MyDrive/Dashboard-BR-CA-Data/exports_all.parquet"
dataset.to_parquet(output_path, engine="pyarrow", index=False)

print("Parquet file saved at:", output_path)

Final concatenated dataset shape: (2562365, 9)
Parquet file saved at: /content/drive/MyDrive/Dashboard-BR-CA-Data/exports_all.parquet
